<a href="https://colab.research.google.com/github/kamat-v/qc-course-materials/blob/main/notebooks/week09_factoring_as_order_finding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import math
import random

#Helper functions:

1. Construct $\mathbb{Z}_N^{\times}$ -- the multiplicative group modulo $N$, which contains all positive integers smaller than $N$ that are coprime with $N$.
2. A *brute-force* order-finding function -- given $a$ and $N$, compute $\text{ord}_N(a)$, the smallest positive integer $r$ such that $a^{r}\equiv 1 (\text{mod } N)$.
[**Note**: This is the part that is classically inefficient, and is handled via a quantum subroutine called Quantum Phase Estimation.]
3. A predicate function that determines whether or not a randomly chosen $a$ satisfies each of the following two conditions:
   * $r=\text{ord}_N(a)$ is even, and
   * $a^{r/2}\not\equiv -1(\text{mod } N)$.
4. A function that estimates, using Monte Carlo sampling, for a given value of $N$ and number of trials, the probability that uniformly sampled $a\in \mathbb{Z}_N^{\times}$ satisfies the two above conditions.

In [ ]:
def multiplicative_group(N):
    """Return list of elements of Z_N*."""
    return [a for a in range(2, N) if math.gcd(a, N) == 1]

def order(a, N):
    """Compute ord_N(a) by brute force."""
    r, x = 1, a % N
    while x != 1:
        x = (x * a) % N
        r += 1
    return r


def is_good(a, N):
    """
    Return True if a is 'good' for Shor's post-processing:
      - r = ord_N(a) is even, and
      - a^{r/2} != -1 mod N.
    """
    r = order(a, N)
    if r % 2 == 1:
        return False
    if pow(a, r // 2, N) == N - 1:   # N - 1 = -1 mod N
        return False
    return True

def monte_carlo(N, trials=10_000):
    """Estimate success probability by sampling."""
    group = multiplicative_group(N)
    successes = sum(1 for _ in range(trials) if is_good(random.choice(group), N))
    return successes / trials

In the code cell below, we estimate, via Monte Carlo sampling, a lower bound on the probability of choosing a "good" $a$, meaning:
1. The order $r=\text{ord}_N(a)$ modulo $N$ is even, and
2. $a^{r/2}\not\equiv -1 (\text{mod } N)$.

For each candidate $N$, we sample multiple $a$ values, and compute the empirical fraction that satisfy these values. The results consistently suggest that $$\text{Pr}[a \text{ is good}]\gtrsim 0.5,$$ thus agreeing with the theoretical guarantee.

As a consequence, repeating the sampling $k=11$ times yields a success probability of at least $$1-(1-0.5)^k\approx 0.9995.$$

In [ ]:
candidates = [
    15,    # 3 * 5
    21,    # 3 * 7
    35,    # 5 * 7
    55,    # 5 * 11
    77,    # 7 * 11
    143,   # 11 * 13
    221,   # 13 * 17
    323,   # 17 * 19
    945,   # 3^3 * 5 * 7
    1001,  # 7 * 11 * 13
    1155,  # 3 * 5 * 7 * 11
    1729,  # 7 * 13 * 19
    3003,  # 3 * 7 * 11 * 13
    3465,  # 3^2 * 5 * 7 * 11
    10405395
]

print(f"{'N':>6} "
      f"{'MC estimate':>14}")
print("-" * 30)

for N in candidates:
    prob_mc = monte_carlo(N, trials=200000)
    # simple factorization label
    print(f"{N:>6}  "
          f"{prob_mc:>11.4f}")


     N    MC estimate
------------------------------
    15       0.8574
    21       0.5449
    35       0.7833
    55       0.7696
    77       0.5094
   143       0.7546
   221       0.9111
   323       0.9416
   945       0.8769
  1001       0.8757
  1155       0.9401
  1729       0.8766
  3003       0.9377
  3465       0.9381


#Finding order (classically) using Shor's algorithm

1. $N$ is assumed odd, composite, and not a prime power.
2. Uniformly sample $a\in \mathbb{Z}_N^{\times}$.
3. If $g=\text{GCD}(a,N)>1$, return $g$.
4. Using a classical brute-force strategy, compute $r=\text{ord}_N(a)$. <-- soon to be **quantumized**!
5. If $r$ is odd, go to step $2$.
6. If $a^{r/2}\equiv -1 (\text{mod } N)$, go to step $2$.
7. Compute $g=\text{GCD }(a^{r/2}-1,N).$
8. Return $g$.

In [ ]:
N=21
print(f"Running the classical Shor's algorithm on N={N}")
ZN=multiplicative_group(N)
steps=0
while True:
    steps+=1
    a=random.choice(ZN)
    r=order(a,N)
    print(f"Step {steps}: We randomly chose {a}")
    if math.gcd(a,N)>1:
        print(f"Lucky! GCD of {a} and {N} is {math.gcd(a,N)}, which is a non-trivial factor of {N}")
        break
    if not is_good(a,N):
        print(f"{a} is no good. It has order {r} and {a}^{int(r/2)} modulo {N} is {a**(r//2)%N}")
        continue
    else:
      print(f"{a} is good. It has order {r} and {a}^{int(r/2)} modulo {N} is {a**(r//2)%N}")
      print(f"Non-trivial factor of {N} found, it is {math.gcd(a**(r//2)-1,N)}")
      break



Running the classical Shor's algorithm on N=10405395
Step 1: We randomly chose 3757372
3757372 is good. It has order 13860 and 3757372^6930 modulo 10405395 is 1760914
Non-trivial factor of 10405395 found, it is 160083
